# LMIP Bronze Layer - Complete Documentation

The Bronze layer is the **raw data lake** of the Labour Market Intelligence Platform. It stores complete, unmodified API responses as immutable snapshots, enabling schema evolution, reprocessing, and forensic analysis.

---

## Architecture Principles

* **Immutability**: Never update or delete raw snapshots (append-only)
* **Schema-on-Read**: Store flexible JSON payloads, enforce schema in silver
* **Full Lineage**: Track every snapshot to its ingestion batch
* **Time Travel**: Reconstruct job posting history from snapshots
* **Reprocessing**: Re-extract data from raw JSON without re-ingesting
* **Payload Preservation**: Store complete API response for maximum flexibility

---

## Pipeline Flow

```
┌─────────────────────────────────────────────────────────────────┐
│                    INGESTION LAYER                              │
│      (ingest_remotive, ingest_arbeitnow + manifest_writer)      │
└────────────────────┬────────────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────────────┐
│  BRONZE LAYER: Raw Data Lake                                    │
├─────────────────────────────────────────────────────────────────┤
│  1. bronze.bronze_job_snapshot                                  │
│     • Raw API responses stored as JSON in payload_json          │
│     • One row per job per batch                                 │
│     • Source-specific payload format (Remotive, Arbeitnow)      │
│     • Immutable: append-only, no updates or deletes             │
│     • Includes payload_hash for deduplication                   │
│                                                                 │
│  2. bronze.bronze_api_response_log                              │
│     • HTTP request/response telemetry                           │
│     • Status codes, response times, retry counts                │
│     • Rate limit tracking                                       │
├─────────────────────────────────────────────────────────────────┤
│  OPTIONAL POST-PROCESSING:                                      │
│  3. bronze.dedupe_tracking (via bronze_dedupe_raw_payload)      │
│     • Identifies duplicate payloads                             │
│     • Non-destructive tracking                                  │
└─────────────────────────────────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────────────┐
│  AUDIT & METADATA LAYER                                         │
├─────────────────────────────────────────────────────────────────┤
│  • audit.audit_pipeline_runs                                    │
│    ➜ Pipeline execution audit (status, counts, runtime)         │
│  • metadata.pipeline_run_control                                │
│    ➜ Pipeline run tracking and control                          │
└─────────────────────────────────────────────────────────────────┘

                          ⬇
                   SILVER LAYER
              (Standardization & Enrichment)
```

---

## Bronze Schema

### bronze.bronze_job_snapshot

**Purpose**: Raw job posting snapshots from all sources

**Schema**:

| Column                 | Type      | Description                                |
| ---------------------- | --------- | ------------------------------------------ |
| `snapshot_id`          | STRING PK | Unique snapshot identifier                 |
| `source_name`          | STRING    | Source system (remotive, arbeitnow, etc.)  |
| `source_job_id`        | STRING    | Job ID from source API                     |
| `batch_id`             | STRING    | Ingestion batch identifier                 |
| `page_number`          | INT       | Page number in paginated response          |
| `page_size`            | INT       | Records per page                           |
| `payload_json`         | STRING    | Complete API response as JSON              |
| `payload_hash`         | STRING    | SHA-256 hash for deduplication             |
| `ingestion_timestamp`  | TIMESTAMP | When snapshot was ingested                 |
| `ingestion_date`       | DATE      | Partition key for ingestion date           |
| `source_status_code`   | INT       | HTTP status code from source               |
| `source_etag`          | STRING    | ETag for change detection                  |

**Partitioning**: Partitioned by `ingestion_date` for query performance

**Primary Key**: `snapshot_id`

**Natural Key**: `source_name + source_job_id + batch_id`

**Deduplication**: Use `payload_hash` to identify duplicates tracked in `bronze.dedupe_tracking`

---

### bronze.dedupe_tracking

**Purpose**: Track duplicate payloads without modifying source table

**Schema**:

| Column                 | Type      | Description                                |
| ---------------------- | --------- | ------------------------------------------ |
| `dedupe_id`            | STRING    | Unique tracking record ID                  |
| `source_table`         | STRING    | Source table being deduplicated            |
| `dedupe_key_hash`      | STRING    | Hash of dedupe key columns                 |
| `first_seen_record_id` | STRING    | Snapshot ID of first occurrence            |
| `duplicate_count`      | INT       | Number of duplicates found                 |
| `first_seen_timestamp` | TIMESTAMP | First ingestion timestamp                  |
| `last_seen_timestamp`  | TIMESTAMP | Most recent duplicate timestamp            |
| `tracking_timestamp`   | TIMESTAMP | When this tracking record was created      |

---

### bronze.bronze_api_response_log

**Purpose**: HTTP request/response telemetry for API monitoring

**Schema**:

| Column               | Type      | Description                             |
| -------------------- | --------- | --------------------------------------- |
| `response_log_id`    | STRING PK | Unique log entry identifier             |
| `source_name`        | STRING    | Source system identifier                |
| `batch_id`           | STRING    | Batch execution identifier              |
| `request_url`        | STRING    | Full request URL                        |
| `http_status_code`   | INT       | HTTP response status                    |
| `retry_count`        | INT       | Number of retry attempts                |
| `rate_limit_hit`     | BOOLEAN   | Was rate limit encountered              |
| `response_time_ms`   | INT       | Response time in milliseconds           |
| `logged_at`          | TIMESTAMP | When the log was created                |

**Use Case**: Monitor API health, track rate limits, debug API issues

---

### audit.audit_pipeline_runs

**Purpose**: Comprehensive pipeline execution audit trail

**Schema**:

| Column             | Type         | Description                          |
| ------------------ | ------------ | ------------------------------------ |
| `audit_run_sk`     | LONG PK      | Surrogate key                        |
| `run_id`           | STRING       | Run identifier (same as batch_id)    |
| `pipeline_name`    | STRING       | Pipeline name                        |
| `environment`      | STRING       | Environment (PROD, DEV, etc.)        |
| `start_time`       | TIMESTAMP    | Run start time                       |
| `end_time`         | TIMESTAMP    | Run end time                         |
| `status`           | STRING       | SUCCESS, FAILED, RUNNING             |
| `rows_read`        | LONG         | Total rows read                      |
| `rows_written`     | LONG         | Total rows written                   |
| `runtime_seconds`  | DECIMAL(18,2)| Execution duration                   |
| `error_message`    | STRING       | Error details if failed              |

**Use Case**: Track pipeline execution history, monitor success rates, diagnose failures

---

### metadata.pipeline_run_control

**Purpose**: Pipeline run tracking and control metadata

**Schema**:

| Column             | Type      | Description                          |
| ------------------ | --------- | ------------------------------------ |
| `run_control_sk`   | LONG PK   | Surrogate key                        |
| `pipeline_name`    | STRING    | Pipeline name                        |
| `batch_id`         | STRING    | Batch execution identifier           |
| `source_name`      | STRING    | Source system name                   |
| `trigger_type`     | STRING    | MANUAL, SCHEDULED, etc.              |
| `scheduled_at`     | TIMESTAMP | When run was scheduled               |
| `started_at`       | TIMESTAMP | When run started                     |
| `ended_at`         | TIMESTAMP | When run completed                   |
| `status`           | STRING    | RUNNING, SUCCESS, FAILED             |
| `operator_user`    | STRING    | User who triggered the run           |

**Use Case**: Control pipeline execution, track run state, manage concurrent runs

---

### Payload Structure by Source

Each source has a different JSON structure in `payload_json`:

#### Remotive Payload

```json
{
  "id": 2090910,
  "url": "https://remotive.com/remote-jobs/software-development/staff-software-engineer-product-belo-horizonte-2090910",
  "title": "Staff Software Engineer, Product (Belo Horizonte)",
  "company_name": "LawnStarter",
  "company_logo": "https://remotive.com/job/2090910/logo",
  "category": "Software Development",
  "tags": [
    "AWS", "backend", "frontend", "php", "react", "security",
    "UI/UX", "AI/ML", "jira", "react native", "documentation",
    "laravel", "Typescript", "people management", "prototyping",
    "engineering management", "marketplace", "github", "REST", "datadog"
  ],
  "job_type": "full_time",
  "publication_date": "2026-06-02T07:53:42",
  "candidate_required_location": "Brazil",
  "salary": "$80k - $100k",
  "description": "<p>This is a remote role for candidates located in Belo Horizonte, Brazil</p>..."
}
```

**Key Fields**:
- `id`: Integer job ID (unique per Remotive)
- `company_name`: Flat string (not nested object)
- `tags`: Array of skill/technology strings
- `publication_date`: ISO 8601 timestamp
- `description`: HTML-formatted job description

---

#### Arbeitnow Payload

```json
{
  "slug": "senior-software-engineer-backend-golang-remote-1234567",
  "company_name": "Tech Corp",
  "title": "Senior Software Engineer - Backend (Golang)",
  "description": "We are looking for a Senior Backend Engineer...",
  "remote": true,
  "url": "https://www.arbeitnow.com/view/senior-software-engineer-backend-golang-remote-1234567",
  "tags": ["golang", "backend", "kubernetes", "microservices"],
  "job_types": ["full-time"],
  "location": "Remote",
  "created_at": 1717326000
}
```

**Key Fields**:
- `slug`: String job ID (unique per Arbeitnow)
- `remote`: Boolean flag for remote work
- `created_at`: Unix timestamp (integer)
- `job_types`: Array (can include "full-time", "part-time", "contract")
- `tags`: Array of skill keywords

---

### Payload Comparison

| Aspect            | Remotive                | Arbeitnow              |
| ----------------- | ----------------------- | ---------------------- |
| **Job ID Type**   | Integer (`id`)          | String (`slug`)        |
| **Date Format**   | ISO 8601 string         | Unix timestamp (int)   |
| **Location**      | String                  | String                 |
| **Remote Flag**   | Implicit (all remote)   | Explicit boolean       |
| **Description**   | HTML                    | Plain text             |
| **Salary**        | String (if provided)    | Not included           |
| **Company Logo**  | URL provided            | Not included           |

---

## Ingestion Notebooks

### ingest_remotive

**Purpose**: Ingest job postings from Remotive API

**Location**: `/LMIP/notebooks/ingestion/ingest_remotive`

**Operations**:
1. Fetches jobs from Remotive API
2. Validates required fields (id, title, company_name, url)
3. Transforms to bronze snapshot format
4. Writes to `bronze.bronze_job_snapshot`
5. Logs API telemetry to `bronze.bronze_api_response_log`
6. Records audit trail in `audit.audit_pipeline_runs`

**Uses**:
- `ingest_common_helpers` for shared logic
- `ingest_manifest_writer` for logging functions

---

### ingest_arbeitnow

**Purpose**: Ingest job postings from Arbeitnow API

**Location**: `/LMIP/notebooks/ingestion/ingest_arbeitnow`

**Operations**:
1. Fetches jobs from Arbeitnow API
2. Validates required fields (slug, title, company_name)
3. Validates data types (created_at=int, remote=bool)
4. Transforms to bronze snapshot format
5. Writes to `bronze.bronze_job_snapshot`
6. Logs API telemetry to `bronze.bronze_api_response_log`
7. Records audit trail in `audit.audit_pipeline_runs`

**Uses**:
- `ingest_common_helpers` for shared logic
- `ingest_manifest_writer` for logging functions

---

### ingest_common_helpers

**Purpose**: Shared ingestion utilities and transformations

**Location**: `/LMIP/notebooks/ingestion/ingest_common_helpers`

**Provides**:
- `generate_payload_hash()`: SHA-256 hash for deduplication
- `generate_snapshot_id()`: UUID generation
- `extract_job_id()`: Source-specific ID extraction
- `ingest_to_bronze()`: Main ingestion orchestration function
- Record transformation routines

---

### ingest_manifest_writer

**Purpose**: Audit and metadata logging functions

**Location**: `/LMIP/notebooks/ingestion/ingest_manifest_writer`

**Provides**:
- `log_api_response()`: Write to `bronze.bronze_api_response_log`
- `start_pipeline_run()`: Initialize run in `metadata.pipeline_run_control`
- `complete_pipeline_run()`: Update run status in `metadata.pipeline_run_control`
- `log_audit_pipeline_run()`: Write to `audit.audit_pipeline_runs`

---

## Bronze Utility Notebooks

### bronze_dedupe_raw_payload

**Purpose**: Identify and track duplicate payloads

**Location**: `/LMIP/notebooks/bronze/bronze_dedupe_raw_payload`

**Operations**:
1. Reads from `bronze.bronze_job_snapshot`
2. Creates hash of dedupe key columns (default: `payload_hash`)
3. Identifies duplicates using window function
4. Writes tracking records to `bronze.dedupe_tracking`

**Parameters**:
- `source_table`: Table to deduplicate (default: `workspace.bronze.bronze_job_snapshot`)
- `dedupe_keys`: Comma-separated columns to hash (default: `payload_hash`)
- `catalog`: Catalog name (default: `workspace`)
- `schema`: Schema name (default: `bronze`)

**Key Features**:
- **Non-destructive**: Does NOT modify source records
- **Idempotent**: Safe to re-run
- **Recent results**: Found 189 duplicate keys

**Note**: This is an **optional** post-processing step, not part of the main ingestion flow.

---

### bronze_replay_backfill

**Purpose**: Replay historical data from bronze layer

**Location**: `/LMIP/notebooks/bronze/bronze_replay_backfill`

**Use Case**: Reprocess bronze data when downstream (silver) logic changes

**Benefits**:
- No API rate limits
- No external API calls needed
- Faster than re-ingestion
- Consistent historical data

---

## Execution Guide

### Running Ingestion Notebooks

Ingestion notebooks are executed independently - no orchestrator notebook exists.

#### Run Remotive Ingestion

```python
# Execute Remotive ingestion
dbutils.notebook.run(
    "/LMIP/notebooks/ingestion/ingest_remotive",
    timeout_seconds=1800
)
```

**Internally, this notebook**:
1. Calls Remotive API
2. Validates and transforms records
3. Writes to `bronze.bronze_job_snapshot`
4. Logs API telemetry via `log_api_response()`
5. Records audit via `log_audit_pipeline_run()`

---

#### Run Arbeitnow Ingestion

```python
# Execute Arbeitnow ingestion
dbutils.notebook.run(
    "/LMIP/notebooks/ingestion/ingest_arbeitnow",
    timeout_seconds=1800
)
```

**Internally, this notebook**:
1. Calls Arbeitnow API
2. Validates record structure and data types
3. Writes to `bronze.bronze_job_snapshot`
4. Logs API telemetry via `log_api_response()`
5. Records audit via `log_audit_pipeline_run()`

---

### Optional Post-Processing

#### Run Deduplication Tracking

```python
# Identify duplicates (optional)
dbutils.notebook.run(
    "/LMIP/notebooks/bronze/bronze_dedupe_raw_payload",
    timeout_seconds=1800,
    arguments={
        "source_table": "workspace.bronze.bronze_job_snapshot",
        "dedupe_keys": "payload_hash",
        "catalog": "workspace",
        "schema": "bronze"
    }
)
```

**This is NOT part of the main ingestion flow** - it's a utility for analyzing duplicate rates.

---

### How Logging Works

Ingestion notebooks use `ingest_manifest_writer` functions internally:

```python
# Example: How ingestion notebooks log API calls
from ingest_manifest_writer import (
    log_api_response,
    start_pipeline_run,
    complete_pipeline_run,
    log_audit_pipeline_run
)

# Log API response telemetry
log_api_response(
    source_name="remotive",
    batch_id=batch_id,
    request_url="https://remotive.com/api/remote-jobs",
    http_status_code=200,
    retry_count=0,
    rate_limit_hit=False,
    response_time_ms=92
)

# Log audit entry
log_audit_pipeline_run(
    run_id=batch_id,
    pipeline_name="bronze_ingestion_remotive",
    status="SUCCESS",
    rows_read=100,
    rows_written=100,
    runtime_seconds=6.03
)
```

**You don't need to call these manually** - ingestion notebooks handle this automatically.

---

### Reprocessing Historical Data

When silver logic changes, reprocess from bronze:

```python
# Replay from bronze without re-calling APIs
dbutils.notebook.run(
    "/LMIP/notebooks/bronze/bronze_replay_backfill",
    timeout_seconds=3600,
    arguments={
        "start_date": "2026-06-01",
        "end_date": "2026-06-03",
        "source_name": "remotive"  # Optional: specific source
    }
)
```

**Benefits**:
- No API rate limits
- Faster than re-ingestion
- Consistent historical data

---

### Querying Bronze Snapshots

#### Latest Snapshot per Job

```sql
-- Get most recent snapshot for each unique job
WITH ranked_snapshots AS (
  SELECT 
    *,
    ROW_NUMBER() OVER (
      PARTITION BY source_name, source_job_id 
      ORDER BY ingestion_timestamp DESC
    ) as rn
  FROM workspace.bronze.bronze_job_snapshot
)
SELECT 
  source_name,
  source_job_id,
  batch_id,
  ingestion_timestamp,
  payload_json
FROM ranked_snapshots 
WHERE rn = 1
ORDER BY ingestion_timestamp DESC
LIMIT 100;
```

#### Check for Duplicates

```sql
-- View duplicate tracking results
SELECT 
  source_table,
  dedupe_key_hash,
  duplicate_count,
  first_seen_timestamp,
  last_seen_timestamp,
  DATEDIFF(DAY, first_seen_timestamp, last_seen_timestamp) as days_between
FROM workspace.bronze.dedupe_tracking
WHERE duplicate_count > 1
ORDER BY duplicate_count DESC, last_seen_timestamp DESC
LIMIT 20;
```

#### Pipeline Execution Audit

```sql
-- View pipeline execution audit history
SELECT 
  run_id as batch_id,
  pipeline_name,
  status,
  start_time,
  end_time,
  rows_read,
  rows_written,
  ROUND(runtime_seconds, 2) as runtime_sec,
  error_message
FROM workspace.audit.audit_pipeline_runs
WHERE pipeline_name LIKE 'bronze_ingestion_%'
ORDER BY start_time DESC
LIMIT 20;
```

#### Daily Ingestion Volumes

```sql
-- Daily snapshot counts by source
SELECT 
  ingestion_date,
  source_name,
  COUNT(*) as snapshot_count,
  COUNT(DISTINCT source_job_id) as unique_jobs,
  COUNT(DISTINCT batch_id) as batch_count
FROM workspace.bronze.bronze_job_snapshot
GROUP BY ingestion_date, source_name
ORDER BY ingestion_date DESC, source_name;
```

#### API Response Telemetry

```sql
-- View API call telemetry
SELECT 
  source_name,
  batch_id,
  http_status_code,
  response_time_ms,
  retry_count,
  rate_limit_hit,
  logged_at
FROM workspace.bronze.bronze_api_response_log
ORDER BY logged_at DESC
LIMIT 20;
```

---

## Best Practices

### Storage Optimization

1. **Partitioning**: Bronze table partitioned by `ingestion_date` for query performance
2. **Compression**: Delta Lake automatically handles compression
3. **Payload Hash**: SHA-256 hash enables efficient deduplication
4. **Metadata Tables**: Keep tracking tables separate from source data

### Query Performance

1. **Avoid SELECT ***: Bronze tables can be large; project only needed columns
2. **Partition Pruning**: Always filter on `ingestion_date` when querying by date range
3. **Dedupe Tracking**: Query `dedupe_tracking` table to identify duplicates
4. **Use payload_hash**: More efficient than comparing full JSON payloads

### Data Quality

1. **Immutability**: Never UPDATE or DELETE bronze snapshots
2. **Append-Only**: All bronze operations are append-only
3. **Deduplication**: Track duplicates in separate table, don't modify source
4. **Audit Logging**: Pipeline runs automatically logged via manifest_writer
5. **API Telemetry**: All API calls automatically logged for monitoring

### Monitoring

1. **Track Pipeline Success**: Monitor `audit.audit_pipeline_runs` for failed runs
2. **Duplicate Rate**: Check `bronze.dedupe_tracking` for duplicate trends
3. **API Health**: Review `bronze.bronze_api_response_log` for API issues
4. **Run Control**: Check `metadata.pipeline_run_control` for run state

---

## Troubleshooting

### Issue: High Duplicate Rate (>20%)

**Symptoms**: Many entries in `dedupe_tracking` table

**Causes**:
1. Ingestion running too frequently (same jobs re-ingested)
2. API returning same data across batches
3. Source job content unchanged (same payload_hash)

**Solutions**:
- Check ingestion schedule (daily is typical for Remotive/Arbeitnow)
- Review API behavior - some sources return full catalog each time
- This is expected behavior for full-refresh APIs (not an error)

**Check Duplicates**:
```sql
-- View duplicate patterns
SELECT 
  dedupe_key_hash,
  duplicate_count,
  DATEDIFF(DAY, first_seen_timestamp, last_seen_timestamp) as days_span
FROM workspace.bronze.dedupe_tracking
WHERE duplicate_count > 5
ORDER BY duplicate_count DESC
LIMIT 20;
```

---

### Issue: Missing Snapshots

**Symptoms**: Expected jobs not in bronze table

**Check**:
1. Did pipeline run complete?
2. Did API return data?
3. Was audit entry created?

**Debug**:
```sql
-- Check pipeline run status
SELECT 
  run_id as batch_id,
  pipeline_name,
  status,
  rows_read,
  rows_written,
  error_message
FROM workspace.audit.audit_pipeline_runs
WHERE run_id = '<batch_id>';

-- Check if records exist
SELECT COUNT(*)
FROM workspace.bronze.bronze_job_snapshot
WHERE batch_id = '<batch_id>';

-- Check API response
SELECT 
  http_status_code,
  response_time_ms,
  retry_count,
  rate_limit_hit
FROM workspace.bronze.bronze_api_response_log
WHERE batch_id = '<batch_id>';
```

---

### Issue: Pipeline Run Failed

**Symptoms**: Pipeline in `audit_pipeline_runs` shows status = 'FAILED'

**Causes**:
1. API timeout or error
2. Network connectivity issue
3. Data validation failure
4. Rate limit hit

**Debug**:
```sql
-- Check failed runs
SELECT 
  run_id,
  pipeline_name,
  status,
  error_message,
  start_time,
  end_time
FROM workspace.audit.audit_pipeline_runs
WHERE status = 'FAILED'
ORDER BY start_time DESC
LIMIT 10;

-- Check API issues for that batch
SELECT 
  source_name,
  http_status_code,
  rate_limit_hit,
  retry_count
FROM workspace.bronze.bronze_api_response_log
WHERE batch_id IN (
  SELECT run_id FROM workspace.audit.audit_pipeline_runs 
  WHERE status = 'FAILED' 
  LIMIT 5
);
```

**Solution**: Review error_message and ingestion notebook logs for details

---

### Issue: API Rate Limits

**Symptoms**: API calls failing with rate limit errors

**Check**:
```sql
-- View rate limit hits
SELECT 
  source_name,
  COUNT(*) as rate_limit_hits,
  MAX(logged_at) as last_hit
FROM workspace.bronze.bronze_api_response_log
WHERE rate_limit_hit = true
GROUP BY source_name
ORDER BY rate_limit_hits DESC;

-- View retry patterns
SELECT 
  source_name,
  retry_count,
  COUNT(*) as occurrences
FROM workspace.bronze.bronze_api_response_log
WHERE retry_count > 0
GROUP BY source_name, retry_count
ORDER BY source_name, retry_count;
```

**Solutions**:
- Reduce ingestion frequency
- Implement exponential backoff
- Contact API provider for rate limit increase

---

### Issue: Large Storage Growth

**Symptoms**: Bronze table size growing rapidly

**Analysis**:
```sql
-- Storage by source and date
SELECT 
  source_name,
  ingestion_date,
  COUNT(*) as snapshot_count,
  SUM(LENGTH(payload_json)) / (1024*1024) as total_mb
FROM workspace.bronze.bronze_job_snapshot
GROUP BY source_name, ingestion_date
ORDER BY ingestion_date DESC, total_mb DESC;
```

**Causes**:
1. High duplicate ingestion (same jobs repeatedly)
2. Large HTML descriptions in payloads
3. No cleanup of old data

**Solutions**:
- Review deduplication tracking
- Consider payload size limits in ingestion
- Implement data retention policy if needed

---

## Data Lineage

### Upstream Dependencies

* **Ingestion Layer**: Source notebooks write to bronze
  - [ingest_remotive](#notebook/1778751970498962) (ingestion)
  - [ingest_arbeitnow](#notebook/1778751970498961) (ingestion)
  - [ingest_common_helpers](#notebook/1778751970498960) (ingestion)
* **API Sources**: External job boards (Remotive, Arbeitnow)

### Downstream Consumers

* **Silver Layer**: Reads from `bronze.bronze_job_snapshot` for standardization
* **Deduplication Analysis**: Uses `bronze.dedupe_tracking` for quality metrics
* **Monitoring Dashboards**: Query `bronze.batch_metadata` for pipeline health
* **Audit & Compliance**: Access `bronze.api_response_logs` and `bronze.job_snapshots`

---

## Bronze Layer Tables Summary

| Schema   | Table                       | Purpose                           | Updated By                     |
| -------- | --------------------------- | --------------------------------- | ------------------------------ |
| bronze   | `bronze_job_snapshot`       | Raw job data                      | Ingestion notebooks            |
| bronze   | `bronze_api_response_log`   | API request/response telemetry    | ingest_manifest_writer         |
| bronze   | `dedupe_tracking`           | Duplicate tracking (optional)     | bronze_dedupe_raw_payload      |
| audit    | `audit_pipeline_runs`       | Pipeline execution audit          | ingest_manifest_writer         |
| metadata | `pipeline_run_control`      | Pipeline run tracking & control   | ingest_manifest_writer         |

---

## Next Steps: Silver Layer

Bronze data flows to the **Silver layer** for:

1. **Standardization**: Parse JSON payloads into structured schema
2. **Normalization**: Company names, job titles, locations
3. **Enrichment**: Skills extraction, sector assignment
4. **Quality Control**: Data quality validation
5. **Deduplication**: Consolidate across sources using `payload_hash`

See [README_SILVER](../silver/README_SILVER) for silver layer documentation.
